In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain import PromptTemplate
from langchain import load_summarize_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader

In [33]:
load_dotenv()

True

In [ ]:
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-20b", api_key=api_key)

In [51]:
# PDF Loader

loader = PyPDFLoader("con.pdf")
docs = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
pdf_docs = text_splitter.split_documents(docs)
pdf_docs

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2025-09-04T14:58:46-07:00', 'moddate': '2025-09-04T14:58:46-07:00', 'source': 'con.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content="BYTMOS CONTRACT AGREEMENT \nThis ContractAgreement is made on 12th July, 2025 between Bytmos LTD. a company of the following \naddress: 8 Kajola Street Ojodu Berger, Lagos State, (hereinafter referred to as “Company”) AND \nEmmanuel Oluwakayode, an Individual of the following address: 1 Olakunle Street Mushin, Lagos \nState (hereinafter referred to as “Contractor”) \nWHEREAS  \n• The Company wishes to retain the services of the Contractor and is of the opinion that the \nContractor is qualified to provide the services stated in this agreement. \n• The Contractor has agreed to provide the services in accordance with the terms and conditions \nherein contained in this agreement. \nTERMS OF AGREEMENT \nThis agreement shall commence o

In [36]:
# Text Loader
loader = TextLoader("ios.txt")
docs = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
text_docs = text_splitter.split_documents(docs)
text_docs

[Document(metadata={'source': 'ios.txt'}, page_content="What BytmosPartner Offers\n\nBytmosPartner is designed for modern professionals who prioritize integrity, quality, and efficiency. Whether you're a freelancer, technician, artisan, consultant, or creative, you'll find tools that help you:"),
 Document(metadata={'source': 'ios.txt'}, page_content='-Sell services seamlessly with clear pricing and easy booking\n-Connect with clients actively searching for your skills\n-Deliver service without complaints through smart tools\n-Maintain integrity in all your professional interactions\n-Grow faster with features built for success\n\nKey Features\n\nProfessional Profile – Build and manage a profile that showcases your skills\n\nService Listings – Present your services clearly with transparent pricing'),
 Document(metadata={'source': 'ios.txt'}, page_content='Service Listings – Present your services clearly with transparent pricing\n\nIn-App Messaging – Communicate with clients directly an

In [53]:
# Web base Loader
loader = WebBaseLoader("https://my.clevelandclinic.org/health/diseases/7104-diabetes")
docs = loader.load_and_split()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
web_docs = text_splitter.split_documents(docs)
web_docs

[Document(metadata={'source': 'https://my.clevelandclinic.org/health/diseases/7104-diabetes', 'title': 'Diabetes: What It Is, Causes, Symptoms, Treatment & Types', 'description': 'Diabetes is a condition that happens when your blood sugar is too high. It develops when your pancreas doesn’t make any insulin, or your body isn’t using it properly.', 'language': 'en'}, page_content='Diabetes: What It Is, Causes, Symptoms, Treatment & TypesLocations:Abu Dhabi|Canada|Florida|London|Nevada|Ohio|800.223.2273|MyChart|Need Help?|Careers|Donate Now|Find a ProviderLocations and DirectionsInstitutes and DepartmentsPatients and VisitorsHealth LibraryFind a ProviderLocations and DirectionsInstitutes and DepartmentsPatients and VisitorsHealth LibraryMenuRequest an AppointmentHome/Health Library/Diseases & Conditions/DiabetesAdvertisementAdvertisementDiabetesDiabetes is a common'),
 Document(metadata={'source': 'https://my.clevelandclinic.org/health/diseases/7104-diabetes', 'title': 'Diabetes: What It 

In [ ]:
chunk_prompt = """
  Summarize the following texts below:
  text: {text}
  Summary:
  """

map_prompt_template = PromptTemplate(input_variables=["text"], template=chunk_prompt)

In [39]:
final_prompt = """
    Summarize the entire texts in well-organized summary spanning 10 pages. Break the summary into paragraphs, 
    each focusing on key points. 
    Book: {text}
"""

final_prompt_template = PromptTemplate(inpput_variables=["text"], template=final_prompt)
final_prompt_template

PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='\n    Summarize the entire texts in well-organized summary spanning 10 pages. Break the summary into paragraphs, \n    each focusing on key points. \n    Book: {text}\n')

In [45]:
summary_chain = load_summarize_chain(llm=llm, chain_type="stuff", prompt=final_prompt_template, verbose=True)

In [54]:
output = summary_chain.run(web_docs)
output




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

    Summarize the entire texts in well-organized summary spanning 10 pages. Break the summary into paragraphs, 
    each focusing on key points. 
    Book: Diabetes: What It Is, Causes, Symptoms, Treatment & TypesLocations:Abu Dhabi|Canada|Florida|London|Nevada|Ohio|800.223.2273|MyChart|Need Help?|Careers|Donate Now|Find a ProviderLocations and DirectionsInstitutes and DepartmentsPatients and VisitorsHealth LibraryFind a ProviderLocations and DirectionsInstitutes and DepartmentsPatients and VisitorsHealth LibraryMenuRequest an AppointmentHome/Health Library/Diseases & Conditions/DiabetesAdvertisementAdvertisementDiabetesDiabetes is a common

& Conditions/DiabetesAdvertisementAdvertisementDiabetesDiabetes is a common condition that affects people of all ages. There are several forms of diabetes. Type 2 is the most common. A combination of treatment strategies can help you manage th

'**Diabetes: A Comprehensive Summary (≈\u202f10\u202fPages)**  \n\n---\n\n### 1. Introduction  \nDiabetes mellitus is a chronic metabolic disorder that affects the body’s ability to regulate blood glucose. It is a global health burden, with an estimated 537\u202fmillion adults worldwide and 37.3\u202fmillion in the United States alone. The condition is characterized by hyperglycaemia resulting from either insufficient insulin production, impaired insulin action, or both. The Cleveland Clinic’s comprehensive guide underscores that, while diabetes is lifelong, it is manageable with a combination of medical therapy, lifestyle modification, and vigilant self‑monitoring.  \n\n---\n\n### 2. Types of Diabetes  \n| Type | Key Features | Typical Age of Onset | Prevalence |\n|------|--------------|----------------------|------------|\n| **Type\u202f2** | Insulin resistance + relative insulin deficiency | Adults (but increasingly children) | 90–95\u202f% of cases |\n| **Prediabetes** | Blood gluc